# Проект 1. Шеринг в ПРАЙМ: стоит ли в него вкладываться

> Как запускать: `Kernel → Restart Kernel and Run All Cells`. Нужен `.env` с `PRIME_DSN`, запуск из окружения репозитория (`uv run jupyter lab`). Ноутбук только читает базу, все числа в выводах считаются кодом.

## 1. Бизнес-вопрос

ПРАЙМ позволяет делить подписку: у «Базового» к владельцу можно подключить до 3 человек, у «Расширенного» — до 5. **Стоит ли компании вкладываться в шеринг (заполнять слоты, расширять их), чтобы удерживать подписчиков и растить аудиторию?**

Вопрос раскладывается на две проверяемые части:

| | Вопрос | Как проверяем | Что будет решением |
|---|---|---|---|
| **H1** | Подписки, у которых заняты все слоты, отменяют реже остальных? | доля отмен на 90–180-й день жизни подписки: «все слоты заняты» против «не все» | если да — поощряем заполнение слотов, если нет — не тратим на это бюджет |
| **Декомпозиция** | За счёт чего растёт число людей, которые пользуются ПРАЙМ (янв → июн 2026)? | рост людей = новые подписки × места на подписку × доля уникальных людей среди мест | показывает, куда направлять бюджет роста |

## 2. Метрики и зерно расчёта

**Что такое одна строка таблицы и почему**

- **H1: одна строка — подписка.** Отмена — это событие подписки, платит тоже подписка, и «заполненность» — свойство подписки. Человек как зерно не подходит: один человек может сидеть в нескольких подписках и попадёт в выборку несколько раз. Платёж как зерно тоже не подходит: он ничего не говорит об отмене.
- **Аудитория: один месяц — три числа** (подписки, занятые места, уникальные люди). Места и людей нельзя смешивать: человек в двух подписках занимает два места, но это один человек. Поэтому ключевая метрика — **люди** (уникальный `client_id`), а места нужны только как промежуточный множитель.

**Определения**

- *Заполнена к N-му дню:* число подключённых участников, вступивших за первые N дней жизни подписки, равно `share_slots` тарифа. Владелец в `subscription_members` не входит (проверяем ниже: участников никогда не больше слотов).
- *Отмена в окне:* `ended_at` попал в интервал от N-го до (N+90)-го дня. Базовый случай N = 90, то есть окно 90–180-й день. Смотрим только подписки, которые дожили до N-го дня, иначе заполненность и «жизнь» перемешиваются.
- *Место* — владелец или участник подписки. *Человек* — уникальный клиент среди всех мест месяца.

**Период.** Для H1 берём подписки, стартовавшие так давно, что у каждой есть полное окно в 180 дней (дату выводим из данных, см. ниже). Для аудитории — январь–июнь 2026, помесячно.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


# Оформление: один акцентный цвет, всё остальное серое, подписи прямо на графиках.
ACC, GREY, DARK = "#C1272D", "#9CA3AF", "#111827"
plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 15, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#E5E7EB", "axes.axisbelow": True,
})
SHORT = {1: "Базовый", 2: "Расширенный"}


def ru(v, d=1, sign=False):
    """Число с запятой вместо точки: 11,8."""
    s = f"{v:+.{d}f}" if sign else f"{v:.{d}f}"
    return s.replace(".", ",").replace("-", "−")


print("подключились как:", q("select current_user as login").iloc[0, 0])

## 3. Проверка данных до расчётов

Сначала смотрим, не врут ли таблицы: сколько подписок, отмен, участников и нет ли строк, которые сломают метрику.

In [ ]:
print("Участники подписок:")
print(q("""
select count(*) as строк,
       count(left_at) as вышли,
       count(*) - count(left_at) as остались
from prime.subscription_members
""").to_string(index=False))

print("\nПодписки: пробные и отменённые")
print(q("""
select is_trial, count(*) as подписок, count(ended_at) as отменено
from prime.subscriptions
group by 1 order by 1
""").to_string(index=False))

print("\nПричины отмен")
print(q("""
select cancel_reason, count(*) as подписок
from prime.subscriptions
where ended_at is not null
group by 1 order by 2 desc
""").to_string(index=False))

print("\nСлоты по тарифам (вариантов должно быть 1: слоты за период не менялись)")
print(q("""
select tariff_id, count(distinct share_slots) as вариантов, max(share_slots) as слотов
from prime.tariffs
group by 1 order by 1
""").to_string(index=False))

In [ ]:
print("Логические проверки (нули — хорошо; ненулевое надо объяснить):")
print(q("""
select count(*) filter (where m.joined_at < s.started_at)                       as вступили_до_старта,
       count(*) filter (where m.left_at < m.joined_at)                          as вышли_до_вступления,
       count(*) filter (where s.ended_at is not null and m.left_at is null)     as в_отменённой_без_выхода,
       count(*) filter (where m.joined_at > s.ended_at)                         as вступили_после_отмены,
       count(*) filter (where m.left_at > s.ended_at)                           as вышли_после_отмены
from prime.subscription_members m
join prime.subscriptions s on s.subscription_id = m.subscription_id
""").T.rename(columns={0: "строк"}))

print("\nУчастников больше лимита слотов (должно быть 0):")
print(q("""
select count(*) as подписок_сверх_лимита
from (select m.subscription_id, count(*) as n
      from prime.subscription_members m group by 1) x
join prime.subscriptions s on s.subscription_id = x.subscription_id
join (select tariff_id, max(share_slots) as share_slots from prime.tariffs group by 1) t
  on t.tariff_id = s.tariff_id
where x.n > t.share_slots
""").to_string(index=False))

**Что важно из проверок (учтено в расчётах ниже):**

- Слоты по тарифам не менялись, участников никогда не больше лимита, а владелец в `subscription_members` не входит. Значит, «заполнена» — это `участников == share_slots`.
- При отмене подписки `left_at` у участников **не проставляется** (у сотен тысяч участников отменённых подписок он пустой). Поэтому «место занято» в месяце определяем по двум датам сразу: подписка ещё не отменена **и** участник ещё не вышел.
- Есть вступления и выходы **после** отмены подписки (десятки тысяч строк). Такие строки в аудиторию не включаем: участник считается только пока подписка жива.

In [ ]:
# Горизонт данных: самое позднее событие в подписках.
hz = q("select max(started_at) as last_start, max(ended_at) as last_end from prime.subscriptions").iloc[0]
last_event = max(hz["last_start"], hz["last_end"])

# Когорта H1: подписки, стартовавшие не позже этой даты, имеют полное окно 180 дней.
COHORT_END = pd.Timestamp("2026-02-20", tz="UTC")
enough = COHORT_END + pd.Timedelta(days=180) <= last_event
print(f"последнее событие в данных: {last_event:%Y-%m-%d}")
print(f"когорта H1: старт до {COHORT_END:%Y-%m-%d}; +180 дней = {COHORT_END + pd.Timedelta(days=180):%Y-%m-%d}")
print("✅ окно наблюдения полное у всех подписок когорты" if enough else "⚠️ у части подписок окно 180 дней неполное — сдвиньте COHORT_END раньше")

## 4. H1: заполненные слоты и отмены

Одна строка — подписка. Выгружаем когорту и вступления участников (в днях от старта подписки), а дальше все варианты расчёта идут одной функцией.

In [ ]:
cohort = q("""
select s.subscription_id, s.tariff_id, sl.share_slots, s.is_trial,
       s.cancel_reason, s.started_at, s.ended_at
from prime.subscriptions s
join (select tariff_id, max(share_slots) as share_slots
      from prime.tariffs group by 1) sl on sl.tariff_id = s.tariff_id
where s.started_at < :cutoff
""", cutoff=COHORT_END.to_pydatetime())
cohort["end_day"] = (cohort["ended_at"] - cohort["started_at"]).dt.total_seconds() / 86400

md = q("""
select m.subscription_id,
       extract(epoch from (m.joined_at - s.started_at)) / 86400.0 as day,
       extract(epoch from (m.left_at   - s.started_at)) / 86400.0 as left_day
from prime.subscription_members m
join prime.subscriptions s on s.subscription_id = m.subscription_id
where s.started_at < :cutoff
""", cutoff=COHORT_END.to_pydatetime()).astype({"day": "float64", "left_day": "float64"})

print("подписок в когорте:", len(cohort), "| ключ уникален:", cohort["subscription_id"].is_unique)
print("вступлений участников:", len(md))

In [ ]:
def churn_gap(N, active_only=False, only_paid=False, voluntary_only=False):
    """Сравнение «все слоты заняты к N-му дню» и «не все» по доле отмен в окне N…N+90 день.

    active_only    — считаем только тех, кто к N-му дню ещё не вышел (а не всех вступивших);
    only_paid      — выкидываем пробные подписки;
    voluntary_only — не считаем отмены из-за неоплаты (payment_failed): это не решение клиента.
    """
    d = cohort
    if only_paid:
        d = d[~d["is_trial"]]
    d = d[d["end_day"].isna() | (d["end_day"] >= N)]            # только дожившие до N-го дня

    joined = md[md["day"] < N]
    if active_only:
        joined = joined[joined["left_day"].isna() | (joined["left_day"] >= N)]
    n = d["subscription_id"].map(joined.groupby("subscription_id").size()).fillna(0)

    full = n == d["share_slots"]
    ended = d["end_day"].notna() & (d["end_day"] < N + 90)
    if voluntary_only:
        ended &= d["cancel_reason"] != "payment_failed"
    ended = ended.astype(float)

    rows = []
    for t in sorted(d["tariff_id"].unique()):
        mask = d["tariff_id"] == t
        a, b = ended[mask & full], ended[mask & ~full]
        se = (a.var() / len(a) + b.var() / len(b)) ** 0.5
        rows.append({"N": N, "tariff": t, "tariff_name": SHORT[t], "full_n": len(a), "rest_n": len(b),
                     "full": a.mean() * 100, "rest": b.mean() * 100,
                     "diff_pp": (a.mean() - b.mean()) * 100, "ci_pp": 1.96 * se * 100})
    return pd.DataFrame(rows)


res90 = churn_gap(90)
print(res90.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True, layout="constrained")
for ax, (_, r) in zip(axes, res90.iterrows()):
    bars = ax.bar(["Не все места\nзаняты", "Все места\nзаняты"], [r["rest"], r["full"]],
                  color=[GREY, ACC], width=0.55)
    ax.bar_label(bars, labels=[ru(r["rest"]) + " %", ru(r["full"]) + " %"],
                 padding=4, fontsize=16, fontweight="bold")
    ax.set_title(f"{r['tariff_name']}  ({r['full_n'] + r['rest_n']:,} подписок)".replace(",", " "), loc="left", fontsize=14)
    verdict = "в пределах погрешности" if abs(r["diff_pp"]) <= r["ci_pp"] else "различие значимо"
    ax.text(0.5, 0.9, f"разница {ru(r['diff_pp'], 1, True)} п.п. — {verdict}", transform=ax.transAxes,
            ha="center", fontsize=12, color="#374151")
    ax.set_ylim(0, 20)
    ax.set_yticks(range(0, 21, 5))
    ax.grid(False)
    ax.tick_params(axis="y", length=0)
axes[0].set_ylabel("Отменили за 90 дней, % подписок")
fig.suptitle("Полные слоты не снижают отмены", fontsize=17, fontweight="bold", x=0.02, ha="left")
plt.show()

**Как читать график.** Красный столбец — подписки, у которых к 90-му дню заняты все слоты, серый — все остальные. Высота — доля тех, кто отменил подписку в следующие 90 дней. Если бы шеринг удерживал, красный столбец был бы заметно ниже серого. Он не ниже.

### Проверка устойчивости

Один порог «90 дней» можно выбрать удачно. Проверяем другие дни, а также варианты определения: только те, кто не вышел, без пробных подписок, без отмен из-за неоплаты.

In [ ]:
th = pd.concat([churn_gap(N) for N in (30, 45, 60, 90)], ignore_index=True)

variants = pd.concat([
    churn_gap(90).assign(вариант="основной"),
    churn_gap(90, active_only=True).assign(вариант="только не вышедшие участники"),
    churn_gap(90, only_paid=True).assign(вариант="без пробных подписок"),
    churn_gap(90, voluntary_only=True).assign(вариант="без отмен из-за неоплаты"),
], ignore_index=True)

print("Разные дни проверки:")
print(th[["N", "tariff_name", "full_n", "rest_n", "full", "rest", "diff_pp", "ci_pp"]].round(2).to_string(index=False))
print("\nРазные определения (N = 90):")
print(variants[["вариант", "tariff_name", "full", "rest", "diff_pp", "ci_pp"]].round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
ax.axhspan(-0.5, 0.5, color="#E5E7EB", zorder=0)
ax.text(0.8, -0.7, "серая полоса — «практически ноль»: ±0,5 п.п.", transform=ax.get_yaxis_transform(),
        ha="center", va="top", fontsize=11, color="#4B5563")
ax.axhline(0, color=DARK, lw=1)
days = [30, 45, 60, 90]
for t, color, off in ((1, GREY, -1.5), (2, ACC, 1.5)):
    g = th[th["tariff"] == t]
    ax.errorbar(g["N"] + off, g["diff_pp"], yerr=g["ci_pp"], fmt="o", ms=10, color=color,
                elinewidth=1.6, capsize=5, label=SHORT[t])
ax.set_xticks(days, [f"{d}-й день" for d in days])
ax.set_xlim(22, 98)
ax.grid(False)
ax.set_xlabel("На какой день жизни подписки смотрим, заполнена ли она")
ax.set_ylabel("Разница в отменах, п.п.\n(все места заняты минус остальные)")
ax.legend(frameon=False, loc="upper left")
ax.set_title("При любом дне проверки разница около нуля", loc="left")
fig.text(0.01, -0.02, "Отрезок — 95 %-интервал. Чем меньше подписок в группе (30-й день, «Расширенный»), тем он шире.",
         fontsize=10.5, color="#4B5563")
plt.show()

In [ ]:
bound = float((res90["diff_pp"].abs() + res90["ci_pp"]).max())
base_rate = float(res90["rest"].mean())
not_sig = lambda df: bool((df["diff_pp"].abs() <= df["ci_pp"]).all())
h1_ok = not_sig(res90) and not_sig(th[th["N"] >= 45]) and not_sig(variants)

lines = [f"- **{r.tariff_name}:** {ru(r.full)} % против {ru(r.rest)} %, разница {ru(r.diff_pp, 2, True)} п.п. (95 %-интервал ±{ru(r.ci_pp, 2)})"
         for r in res90.itertuples()]
if h1_ok:
    verdict = (f"**H1 не подтвердилась.** Подписки с заполненными слотами отменяют так же часто, как остальные. "
               f"Данные исключают эффект сильнее примерно {ru(bound)} п.п. при базовой доле отмен около {ru(base_rate)} %. "
               f"Вывод не меняется при проверке на 45–90-й день, без пробных подписок и без отмен из-за неоплаты.")
else:
    sig = variants[variants["diff_pp"].abs() > variants["ci_pp"]]
    verdict = ("**В части вариантов разница значима** (" + ", ".join(sorted(set(sig["вариант"] + " / " + sig["tariff_name"]))) +
               "). Смотрите таблицы выше: вывод про слоты надо формулировать осторожно.")
display(Markdown("### Промежуточный вывод по H1\n\n" + "\n".join(lines) + "\n\n" + verdict +
                 "\n\nОграничение: слоты никто не менял и клиентов случайно не распределяли, поэтому это **связь, а не эксперимент**."))

## 5. Откуда растёт аудитория: факторная декомпозиция

Ключевой показатель — **число людей**, которые пользуются ПРАЙМ в месяце (владельцы и подключённые участники, каждый человек один раз). Раскладываем его в произведение трёх множителей:

$$\text{люди} \;=\; \underbrace{\text{подписки}}_{N}\;\times\;\underbrace{\tfrac{\text{места}}{\text{подписки}}}_{s\ \text{— глубина шеринга}}\;\times\;\underbrace{\tfrac{\text{люди}}{\text{места}}}_{o\ \text{— уникальность}}$$

- **N** — сколько подписок активно в месяце (подписка активна, если стартовала до конца месяца и не отменена до его начала);
- **s** — сколько мест в среднем занимает подписка (владелец плюс активные участники). Это и есть «глубина шеринга»;
- **o** — какая доля мест приходится на уникальных людей. Если человек сидит в двух подписках, мест два, а человек один: чем меньше `o`, тем больше пересечений.

Участник считается местом в месяце, если он вступил до конца месяца, подписка не отменена до его начала и сам участник не вышел.

In [ ]:
aud = q("""
with mo as (
    select ms, ms + interval '1 month' as me
    from generate_series('2026-01-01'::timestamptz, '2026-06-01'::timestamptz, interval '1 month') as g(ms)),
act as (
    select mo.ms, mo.me, s.subscription_id, s.owner_client_id, s.ended_at
    from mo
    join prime.subscriptions s
      on s.started_at < mo.me and (s.ended_at is null or s.ended_at >= mo.ms)),
mem as (
    select a.ms, m.client_id
    from act a
    join prime.subscription_members m
      on m.subscription_id = a.subscription_id
     and m.joined_at < a.me
     and (a.ended_at is null or m.joined_at <= a.ended_at)
     and coalesce(least(m.left_at, a.ended_at), 'infinity'::timestamptz) >= a.ms),
n_subs   as (select ms, count(*) as subs    from act group by 1),
n_mem    as (select ms, count(*) as members from mem group by 1),
n_people as (
    select ms, count(*) as people
    from (select ms, owner_client_id as client_id from act
          union
          select ms, client_id from mem) u
    group by 1)
select to_char(a.ms, 'YYYY-MM') as month, a.subs, b.members, c.people
from n_subs a
join n_mem b using (ms)
join n_people c using (ms)
order by 1
""").set_index("month")

aud["seats"] = aud["subs"] + aud["members"]          # владелец + активные участники
aud["s"] = aud["seats"] / aud["subs"]                # мест на подписку
aud["o"] = aud["people"] / aud["seats"]              # доля уникальных людей среди мест
aud["seats_per_person"] = aud["seats"] / aud["people"]
print(aud.round(3).to_string())
assert aud["people"].le(aud["seats"]).all(), "людей не может быть больше, чем мест"

In [ ]:
MONTHS = {"01": "янв", "02": "фев", "03": "мар", "04": "апр", "05": "май", "06": "июн"}
x = np.arange(len(aud))
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
ax.plot(x, aud["seats"] / 1e3, color=GREY, lw=3.5, marker="o", ms=8)
ax.plot(x, aud["people"] / 1e3, color=ACC, lw=3.5, marker="o", ms=8)
ax.text(x[-1] + 0.15, aud["seats"].iloc[-1] / 1e3, f"места\n{aud['seats'].iloc[-1] / 1e3:,.0f} тыс.".replace(",", " "),
        color="#4B5563", va="center", fontsize=13, fontweight="bold")
ax.text(x[-1] + 0.15, aud["people"].iloc[-1] / 1e3, f"люди\n{aud['people'].iloc[-1] / 1e3:,.0f} тыс.".replace(",", " "),
        color=ACC, va="center", fontsize=13, fontweight="bold")
ax.set_xticks(x, [MONTHS[m[-2:]] for m in aud.index])
ax.set_xlim(-0.3, len(x) + 0.3)
ax.set_ylim(0, aud["seats"].max() / 1e3 * 1.15)
ax.set_ylabel("тысяч, 2026 год")
ax.set_title(f"Мест в {ru(aud['seats_per_person'].iloc[-1])} раза больше, чем людей: аудиторию считаем людьми", loc="left")
plt.show()

In [ ]:
def chain(f0, f1, order):
    """Подставляем множители из f1 по одному в заданном порядке; вклад шага — прирост произведения."""
    cur, out = dict(f0), {}
    prev = np.prod(list(cur.values()))
    for k in order:
        cur[k] = f1[k]
        now = np.prod(list(cur.values()))
        out[k] = now - prev
        prev = now
    return out


first, last = aud.iloc[0], aud.iloc[-1]
f0 = {"N": first["subs"], "s": first["s"], "o": first["o"]}
f1 = {"N": last["subs"], "s": last["s"], "o": last["o"]}
dP = last["people"] - first["people"]

eff = chain(f0, f1, ("N", "s", "o"))                      # основной порядок: подписки → глубина → уникальность
assert abs(sum(eff.values()) - dP) < 1e-6, "вклады должны складываться в прирост людей"

# Результат не должен зависеть от порядка подстановки: проверяем все 6 порядков.
orders = pd.DataFrame([{**{"порядок": " → ".join(p)}, **{k: v / 1e3 for k, v in chain(f0, f1, p).items()}}
                       for p in itertools.permutations(("N", "s", "o"))])
print(f"Прирост людей янв → июн: {dP:,.0f}".replace(",", " "))
print("\nВклад факторов, тыс. человек, при разном порядке подстановки:")
print(orders.round(1).to_string(index=False))
spread = orders[["N", "s", "o"]].agg(["min", "max"]).round(1)
same_rank = orders[["N", "s", "o"]].abs().idxmax(axis=1).nunique() == 1
print("\nРазброс по порядкам, тыс.:")
print(spread.to_string())

In [ ]:
NAMES = {"N": "Новые\nподписки", "s": "Больше людей\nна подписку", "o": "Люди в\nнескольких\nподписках"}
vals = [eff["N"] / 1e3, eff["s"] / 1e3, eff["o"] / 1e3]
total = dP / 1e3
starts = [0, vals[0], vals[0] + vals[1]]
main = max(eff, key=lambda k: abs(eff[k]))

fig, ax = plt.subplots(figsize=(10, 5.2), layout="constrained")
pad = max(abs(total), max(abs(v) for v in vals)) * 0.03
for i, (v, st) in enumerate(zip(vals, starts)):
    color = ACC if v >= 0 else GREY
    ax.bar(i, v, bottom=st, color=color, width=0.6)
    top = max(st, st + v)
    ax.text(i, top + pad, ru(v, 1, True) + " тыс.", ha="center", va="bottom", fontsize=15, fontweight="bold")
ax.bar(3, total, color=DARK, width=0.6)
ax.text(3, total + pad, ru(total, 1, True) + " тыс.", ha="center", va="bottom", fontsize=15, fontweight="bold")
ax.axhline(0, color=DARK, lw=1)
ax.set_xticks(range(4), [NAMES["N"], NAMES["s"], NAMES["o"], "Итого:\nприрост людей"])
ax.set_ylabel("тысяч человек, январь → июнь 2026")
ax.set_ylim(min(0, min(starts[2] + vals[2], 0)) - pad * 4, max(starts[2], starts[1] + vals[1], total) * 1.18)
ax.grid(False)
ax.tick_params(axis="y", length=0)
ax.set_title(f"Аудитория выросла на {ru(total)} тыс. человек: главный вклад — {NAMES[main].replace(chr(10), ' ').lower()}", loc="left")
plt.show()

In [ ]:
g = dP / first["people"] * 100
EXPL = {
    "N": "Бюджет роста — на привлечение **новых подписок**: именно они дают прирост людей.",
    "s": "Резерв роста — **глубина шеринга**: чем больше людей на подписку, тем больше аудитория.",
    "o": "Прирост определяет **уникальность людей**: растёт или падает доля пересечений.",
}
rec_h1 = ("**Не вкладывать бюджет в заполнение и расширение слотов ради удержания:** заполненные подписки отменяют не реже остальных."
          if h1_ok else
          "**По слотам нужен осторожный A/B-тест:** в части вариантов различие в отменах значимо, но это связь, а не причинность.")
rec_growth = EXPL[main]
order_note = (f"Размер вкладов слегка зависит от порядка подстановки (у подписок от {ru(spread.loc['min', 'N'])} до {ru(spread.loc['max', 'N'])} тыс. "
              "по шести порядкам), но главный фактор при любом порядке один и тот же." if same_rank else
              "Главный фактор зависит от порядка подстановки, поэтому выводы про вклады надо читать осторожно.")
rec_overlap = (f"**Метрику аудитории считать в людях, а не в местах:** места завышают аудиторию в {ru(last['seats_per_person'], 2)} раза, "
               f"а пересечения (человек в нескольких подписках) «съели» {ru(abs(eff['o'] / 1e3))} тыс. прироста. "
               "Похоже, часть подключённых уже клиенты ПРАЙМ, и новых людей шеринг приводит меньше, чем кажется по местам."
               if eff["o"] < 0 else
               "**Метрику аудитории всё равно считать в людях, а не в местах:** места завышают аудиторию "
               f"в {ru(last['seats_per_person'], 2)} раза.")

display(Markdown(f"""
## 6. Вывод и рекомендация

**Вопрос:** стоит ли вкладываться в шеринг, чтобы удерживать подписчиков и растить аудиторию?

**1. Удержание (H1).** {verdict}

**2. Рост аудитории.** С января по июнь 2026 число людей, пользующихся ПРАЙМ, выросло на {ru(total)} тыс. ({ru(g)} %):
новые подписки дали {ru(eff['N'] / 1e3, 1, True)} тыс., глубина шеринга {ru(eff['s'] / 1e3, 1, True)} тыс.,
пересечения (люди в нескольких подписках) {ru(eff['o'] / 1e3, 1, True)} тыс. {order_note}

**Что делать**

1. {rec_h1}
2. {rec_growth}
3. {rec_overlap}

**Чего расчёт не доказывает.** Это наблюдения, а не эксперимент: клиентов в «полные» и «неполные» подписки никто случайно не распределял, а слоты за период не менялись. Деньги (выручка, цена, платежи) здесь не анализировались; следующий шаг — проверить, как подорожание тарифов повлияло на выручку, и отдельно оценить отмены из-за цены.
"""))

## 7. Числа для записки

Эта ячейка печатает ключевые числа одним блоком, чтобы перенести их в `ЗАПИСКА-ПРОЕКТ-1.md`.

In [ ]:
print("H1, N = 90 (доля отмен в окне 90–180 дней, %):")
print(res90[["tariff_name", "full_n", "rest_n", "full", "rest", "diff_pp", "ci_pp"]].round(2).to_string(index=False))
print(f"\nИсключаемый эффект: больше ≈ {bound:.2f} п.п.; устойчивость к порогам и вариантам: {h1_ok}")
print(f"\nЛюди: январь {first['people']:,.0f} → июнь {last['people']:,.0f}, прирост {dP:,.0f} ({g:.1f} %)".replace(",", " "))
print("Вклады, тыс. человек:", {k: round(float(v) / 1e3, 1) for k, v in eff.items()})
print(f"Мест на человека: январь {first['seats_per_person']:.2f} → июнь {last['seats_per_person']:.2f}")